In [5]:
# 0) 参数配置 + 自动安装依赖 + 读取数据表
import importlib.util
import subprocess
import sys
from pathlib import Path
from datetime import datetime

def ensure_package(pip_name: str, import_name: str | None = None):
    import_name = import_name or pip_name
    if importlib.util.find_spec(import_name) is None:
        print(f"正在安装依赖: {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

ensure_package("pandas")
ensure_package("openpyxl")
ensure_package("moviepy")
ensure_package("faster-whisper", "faster_whisper")

import pandas as pd

project_root = Path("/Users/wangbin/data/jupyter/实战/爬网页/微博b站抖音小红书知乎/小红书-1/视频下载")
xlsx_path = project_root / "图文记录表" / "2小红书数据全-0.xlsx"
assert xlsx_path.exists(), f"数据表不存在: {xlsx_path}"

# 小样本验证：建议先 1~3；全量执行时改为 None
max_files = None

# 可配置项
media_type_col = "媒体类型"
media_rel_path_col = "媒体文件路径"
media_abs_path_col = "媒体文件绝对路径"
target_media_type = "视频"
transcript_col = "视频转文字"

df = pd.read_excel(xlsx_path)
assert media_type_col in df.columns, f"缺少列: {media_type_col}"
assert media_rel_path_col in df.columns or media_abs_path_col in df.columns, "缺少媒体路径列"

if transcript_col not in df.columns:
    df[transcript_col] = ""

video_mask = df[media_type_col].astype(str).str.strip().eq(target_media_type)
video_df = df[video_mask].copy()

if max_files is not None:
    video_df = video_df.head(max_files)

output_root = project_root / "图文记录表" / "transcribe_output"
audio_dir = output_root / "wav"
text_dir = output_root / "txt"
segment_dir = output_root / "segments"
for d in (output_root, audio_dir, text_dir, segment_dir):
    d.mkdir(parents=True, exist_ok=True)

run_tag = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

print("数据表:", xlsx_path)
print("总行数:", len(df))
print("视频行数:", int(video_mask.sum()))
print("本次处理数:", len(video_df))
print("输出目录:", output_root)

数据表: /Users/wangbin/data/jupyter/实战/爬网页/微博b站抖音小红书知乎/小红书-1/视频下载/图文记录表/2小红书数据全-0.xlsx
总行数: 1220
视频行数: 339
本次处理数: 339
输出目录: /Users/wangbin/data/jupyter/实战/爬网页/微博b站抖音小红书知乎/小红书-1/视频下载/图文记录表/transcribe_output


In [6]:
# 1) 定义函数（定位视频 / 提取音频 / 转写）
import hashlib
import json
import re

try:
    from moviepy import VideoFileClip
except Exception:
    from moviepy.editor import VideoFileClip

from faster_whisper import WhisperModel

# 模型可选: tiny / base / small / medium / large-v3
model_size = "small"
model = WhisperModel(model_size, device="auto", compute_type="int8")

def resolve_media_dir(row) -> Path | None:
    rel_val = str(row.get(media_rel_path_col, "")).strip()
    abs_val = str(row.get(media_abs_path_col, "")).strip()

    if rel_val and rel_val.lower() != "nan":
        rel_path = Path(rel_val)
        return rel_path if rel_path.is_absolute() else project_root / rel_path

    if abs_val and abs_val.lower() != "nan":
        return Path(abs_val)

    return None

def resolve_video_file(row):
    media_dir = resolve_media_dir(row)
    if media_dir is None:
        return None, "媒体目录为空"
    if not media_dir.exists():
        return None, f"媒体目录不存在: {media_dir}"

    cands = sorted(media_dir.glob("*.mp4"))
    if cands:
        return cands[0], ""

    return None, f"未找到 mp4: {media_dir}"

def build_safe_stem(video_path: Path) -> str:
    # 优先用目录名（通常是 note_id），避免中文超长文件名引发问题
    base = video_path.parent.name if video_path.parent.name else video_path.stem
    base = re.sub(r"[^\w\u4e00-\u9fff-]+", "_", base).strip("_")
    if not base:
        base = "video"
    digest = hashlib.md5(str(video_path).encode("utf-8")).hexdigest()[:8]
    return f"{base[:64]}_{digest}"

def extract_audio(video_path: Path, audio_path: Path):
    with VideoFileClip(str(video_path)) as clip:
        if clip.audio is None:
            raise ValueError("该视频没有音轨，无法转写")
        clip.audio.write_audiofile(
            str(audio_path),
            fps=16000,
            nbytes=2,
            codec="pcm_s16le",
            ffmpeg_params=["-ac", "1"],
            logger=None,
        )

def transcribe_audio(audio_path: Path, use_vad: bool):
    segments, info = model.transcribe(
        str(audio_path),
        language="zh",
        vad_filter=use_vad,
        beam_size=5,
        condition_on_previous_text=False,
        temperature=0,
        word_timestamps=False,
    )

    rows = []
    texts = []
    for seg in segments:
        text = seg.text.strip()
        if not text:
            continue
        rows.append({
            "start": round(seg.start, 2),
            "end": round(seg.end, 2),
            "text": text,
        })
        texts.append(text)

    return rows, texts, info

print(f"模型已加载: {model_size}")

模型已加载: small


In [ ]:
# 2) 批量处理视频（支持断点续跑）
import csv

summary_rows = []
transcript_by_row_index = {}

total = len(video_df)
max_retries = 2

for idx, (row_index, row) in enumerate(video_df.iterrows(), start=1):
    video_path, err = resolve_video_file(row)
    if video_path is None:
        summary_rows.append({
            "row_index": int(row_index),
            "video_name": "",
            "status": "error",
            "segments": 0,
            "language": "",
            "language_probability": "",
            "vad_used": "",
            "retry_count": 0,
            "audio_path": "",
            "txt_path": "",
            "segments_json_path": "",
            "error": err,
        })
        print(f"[{idx}/{total}] 失败: 行 {row_index} -> {err}")
        continue

    safe_stem = build_safe_stem(video_path)
    audio_path = audio_dir / f"{safe_stem}.wav"
    txt_path = text_dir / f"{safe_stem}.txt"
    json_path = segment_dir / f"{safe_stem}.json"

    # 断点续跑：若已存在转写文本，则直接读取
    if txt_path.exists():
        full_text = txt_path.read_text(encoding="utf-8")
        transcript_by_row_index[row_index] = full_text
        summary_rows.append({
            "row_index": int(row_index),
            "video_name": video_path.name,
            "status": "cached",
            "segments": "",
            "language": "",
            "language_probability": "",
            "vad_used": "",
            "retry_count": 0,
            "audio_path": str(audio_path),
            "txt_path": str(txt_path),
            "segments_json_path": str(json_path),
            "error": "",
        })
        print(f"[{idx}/{total}] 跳过(缓存): {video_path.name}")
        continue

    ok = False
    last_error = ""

    for attempt in range(1, max_retries + 1):
        try:
            extract_audio(video_path, audio_path)

            rows, texts, info = transcribe_audio(audio_path, use_vad=True)
            vad_used = True
            if not rows:
                rows, texts, info = transcribe_audio(audio_path, use_vad=False)
                vad_used = False

            full_text = "\n".join(texts).strip()
            txt_path.write_text(full_text, encoding="utf-8")
            json_path.write_text(
                json.dumps(rows, ensure_ascii=False, indent=2),
                encoding="utf-8",
            )

            transcript_by_row_index[row_index] = full_text
            summary_rows.append({
                "row_index": int(row_index),
                "video_name": video_path.name,
                "status": "ok",
                "segments": len(rows),
                "language": getattr(info, "language", ""),
                "language_probability": round(getattr(info, "language_probability", 0.0), 6),
                "vad_used": vad_used,
                "retry_count": attempt - 1,
                "audio_path": str(audio_path),
                "txt_path": str(txt_path),
                "segments_json_path": str(json_path),
                "error": "",
            })
            print(f"[{idx}/{total}] 完成: {video_path.name} -> 片段 {len(rows)}")
            ok = True
            break
        except Exception as e:
            last_error = str(e)
            print(f"[{idx}/{total}] 重试 {attempt}/{max_retries}: {video_path.name} -> {last_error}")

    if not ok:
        summary_rows.append({
            "row_index": int(row_index),
            "video_name": video_path.name,
            "status": "error",
            "segments": 0,
            "language": "",
            "language_probability": "",
            "vad_used": "",
            "retry_count": max_retries,
            "audio_path": str(audio_path),
            "txt_path": str(txt_path),
            "segments_json_path": str(json_path),
            "error": last_error,
        })
        print(f"[{idx}/{total}] 最终失败: {video_path.name}")

summary_json_path = output_root / f"batch_summary_{run_tag}.json"
summary_csv_path = output_root / f"batch_summary_{run_tag}.csv"

summary_json_path.write_text(
    json.dumps(summary_rows, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

fieldnames = [
    "row_index",
    "video_name",
    "status",
    "segments",
    "language",
    "language_probability",
    "vad_used",
    "retry_count",
    "audio_path",
    "txt_path",
    "segments_json_path",
    "error",
]
with summary_csv_path.open("w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(summary_rows)
ok_count = sum(1 for row in summary_rows if row["status"] in {"ok", "cached"})
fail_count = len(summary_rows) - ok_count
print("\n===== 批量转写完成 =====")
print(f"总数: {len(summary_rows)} | 成功/缓存: {ok_count} | 失败: {fail_count}")
print(f"汇总(JSON): {summary_json_path}")
print(f"汇总(CSV): {summary_csv_path}")

[1/339] 重试 1/2: 69c8017b000000001a02f0b0.mp4 -> 该视频没有音轨，无法转写
[1/339] 重试 2/2: 69c8017b000000001a02f0b0.mp4 -> 该视频没有音轨，无法转写
[1/339] 最终失败: 69c8017b000000001a02f0b0.mp4
[2/339] 跳过(缓存): 69c7a760000000001a032688.mp4
[3/339] 跳过(缓存): 69c736f10000000023023f42.mp4
[4/339] 完成: 69c6bd0f000000002103a531.mp4 -> 片段 18
[5/339] 重试 1/2: 69c611b1000000001a027769.mp4 -> 该视频没有音轨，无法转写
[5/339] 重试 2/2: 69c611b1000000001a027769.mp4 -> 该视频没有音轨，无法转写
[5/339] 最终失败: 69c611b1000000001a027769.mp4
[6/339] 完成: 69c53027000000001a028ad3.mp4 -> 片段 11
[7/339] 完成: 69c4ff66000000001d01edac.mp4 -> 片段 1
[8/339] 完成: 69c4e1b4000000002800829b.mp4 -> 片段 24
[9/339] 重试 1/2: 69c4d105000000002200f8c1.mp4 -> 该视频没有音轨，无法转写
[9/339] 重试 2/2: 69c4d105000000002200f8c1.mp4 -> 该视频没有音轨，无法转写
[9/339] 最终失败: 69c4d105000000002200f8c1.mp4
[10/339] 完成: 69c4ac14000000002301415f.mp4 -> 片段 16
[11/339] 完成: 69c3c3fa000000002200f2d6.mp4 -> 片段 18
[12/339] 完成: 69c3b777000000002102dfd8.mp4 -> 片段 8
[13/339] 重试 1/2: 69c3b4f7000000002103a06b.mp4 -> 该视频没有音轨，无法转写
[1

In [8]:
# 3) 回写新增列并导出结果表
for row_index, text in transcript_by_row_index.items():
    df.at[row_index, transcript_col] = text

output_xlsx_path = xlsx_path.with_name(f"{xlsx_path.stem}_含视频转文字.xlsx")
df.to_excel(output_xlsx_path, index=False)

filled_count = int(df[transcript_col].astype(str).str.strip().ne("").sum())
print("已新增列:", transcript_col)
print("已填充文本行数:", filled_count)
print("输出文件:", output_xlsx_path)
print("\n提示: 当前默认 max_files=3，仅小样本验证。要全量处理请把第1格的 max_files 改为 None 后，从第2格重新执行。")

已新增列: 视频转文字
已填充文本行数: 193
输出文件: /Users/wangbin/data/jupyter/实战/爬网页/微博b站抖音小红书知乎/小红书-1/视频下载/图文记录表/2小红书数据全-0_含视频转文字.xlsx

提示: 当前默认 max_files=3，仅小样本验证。要全量处理请把第1格的 max_files 改为 None 后，从第2格重新执行。
